# 🧠 AURA - Fine-Tuning de Qwen 2.5 Coder con Dataset de Destilación
Este cuaderno entrena tu propio modelo de inteligencia artificial usando la GPU gratuita de Google Colab (NVIDIA T4).

**Tiempo estimado:** ~7 a 10 minutos.
**Resultado:** Un archivo `.gguf` cuantizado (Q4_K_M) listo para reemplazar en tu PC y usar 100% offline.

In [ ]:
# PASO 1: Instalar Unsloth (Acelerador 4x de Fine-Tuning con bajo consumo de VRAM)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets trl peft accelerate bitsandbytes

In [ ]:
# PASO 3: Cargar y formatear tu dataset de destilacion
import os
import json
from datasets import Dataset

dataset_file = "aura_finetune_chatml.jsonl" if os.path.exists("aura_finetune_chatml.jsonl") else ("distillation_dataset.jsonl" if os.path.exists("distillation_dataset.jsonl") else "")
if not dataset_file or not os.path.exists(dataset_file):
    from google.colab import files
    print("Selecciona tu archivo aura_finetune_chatml.jsonl desde tu PC:")
    uploaded = files.upload()
    dataset_file = list(uploaded.keys())[0]

system_instruction = (
    "Eres AURA CORE, un asistente de ingenieria de software autonomo y local en Windows 10. "
    "Cuentas con herramientas locales para explorar directorios, leer codigo, escribir scripts de Python, "
    "ejecutar comandos de PowerShell y monitorizar bots de trading cuantitativo. Responde con soluciones tecnicas completas y codigo funcional."
)

formatted_data = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            item = json.loads(line)
            msgs = item.get("messages", [])
            if len(msgs) >= 2:
                if len(msgs) == 2:
                    chat = [
                        {"role": "system", "content": system_instruction},
                        {"role": "user", "content": msgs[0]["content"]},
                        {"role": "assistant", "content": msgs[1]["content"]}
                    ]
                else:
                    chat = msgs
                text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
                formatted_data.append({"text": text})
        except Exception:
            pass

train_dataset = Dataset.from_list(formatted_data)
print(f"Se prepararon con exito {len(train_dataset)} pares de entrenamiento de elite.")

In [ ]:
# PASO 3: Cargar y formatear tu archivo 'distillation_dataset.jsonl'
import os
import json
from datasets import Dataset

dataset_file = "distillation_dataset.jsonl"
if not os.path.exists(dataset_file):
    from google.colab import files
    print("📂 Selecciona tu archivo 'distillation_dataset.jsonl' desde tu PC:")
    uploaded = files.upload()
    dataset_file = list(uploaded.keys())[0]

system_instruction = (
    "Eres AURA CORE, un asistente de ingeniería de software autónomo y local en Windows 10. "
    "Cuentas con herramientas locales para explorar directorios, leer código, escribir scripts de Python, "
    "ejecutar comandos de PowerShell y monitorizar bots de trading cuantitativo. Responde con soluciones técnicas completas y código funcional."
)

formatted_data = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            item = json.loads(line)
            msgs = item.get("messages", [])
            if len(msgs) >= 2:
                chat = [
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": msgs[0]["content"]},
                    {"role": "assistant", "content": msgs[1]["content"]}
                ]
                text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
                formatted_data.append({"text": text})
        except Exception:
            pass

train_dataset = Dataset.from_list(formatted_data)
print(f"✅ Se prepararon con éxito {len(train_dataset)} pares de entrenamiento de élite.")

In [ ]:
# PASO 4: Entrenar el modelo con SFTTrainer (Supervised Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # ~2 a 3 épocas completas con el dataset
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "aura_training_outputs",
    ),
)

print("🚀 Iniciando entrenamiento LoRA...")
trainer_stats = trainer.train()
print("🎉 ¡Entrenamiento completado con éxito!")

In [ ]:
# PASO 5: Exportar automáticamente a formato GGUF (Q4_K_M) y descargar a tu PC
import os
import glob
from google.colab import files

print("⚙️ Exportando modelo a formato GGUF cuantizado (Q4_K_M)... Esto toma unos 2 minutos.")
model.save_pretrained_gguf("aura_qwen_trained", tokenizer, quantization_method = "q4_k_m")

gguf_candidates = glob.glob("aura_qwen_trained/*.gguf") or glob.glob("*q4_k_m*.gguf") or glob.glob("*.gguf")
if gguf_candidates:
    target_file = gguf_candidates[0]
    print(f"📥 Descargando {target_file} a tu carpeta de Descargas...")
    files.download(target_file)
    print("\nInstrucciones para tu PC:")
    print("1. Mueve el archivo descargado a la carpeta: IA/models/")
    print("2. Reinicia tu servidor local para disfrutar de tu IA 100% personalizada y sin restricciones.")
else:
    print("No se encontró el archivo .gguf generado automáticamente. Revisa los archivos en la barra lateral de Colab.")